### ⚒ Setup

In [ ]:
# pip で最小 dependencies を install
# 通常利用では repository Dockerfile を推奨します。
!pip install cyipopt "pydmf>=1.2.1" "orb-models>=0.7.0" "sella>=v2.4.2" "ase>=3.28.0" "numpy>=2.4.6" "scipy>=1.17.1" "pandas>=3.0.3" "matplotlib>=3.10.9" "seaborn>=0.13.2" "rmsd>=1.6.5" "pillow>=12.2.0"
# memory issue を避けるため JAX GPU preallocation を無効化
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"


In [ ]:
# tblite を source から install
# 1. build tools と dependencies を install
!pip install meson ninja toml

# 2. tblite repository を clone
%cd /content
!git clone --depth 1 https://github.com/tblite/tblite.git /opt/tblite

# 3. tblite binary と library を build / install
%cd /opt/tblite
!meson setup _build --prefix=/usr/local -Dpython=true
!meson compile -C _build
!meson install -C _build
%cd /content

# 4. Python bindings を install
!pip install /opt/tblite/python

# 5. CFFI が shared library を見つけられるよう library path を更新
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/lib:/usr/local/lib/x86_64-linux-gnu'

In [ ]:
# PySCF と GPU4PySCF を install (Colab compatible)
!pip install gpu4pyscf-cuda12x
!pip install "pyscf>=2.13.0"
!python -m cupyx.tools.install_library --cuda 12.x --library cutensor


In [ ]:
# MolScout を取得
!git clone https://github.com/hikuram/MolScout.git
!cp -r MolScout/core/* .

In [ ]:
# matplotlib を設定
import matplotlib
import subprocess
import shutil
import logging

# 1. warnings を抑制
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# 2. Microsoft core fonts (Arial を含む) を install
install_cmd = """
echo "ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true" | debconf-set-selections && \
apt-get update -qq && \
apt-get install -y -qq ttf-mscorefonts-installer
"""
subprocess.run(install_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. 古い Matplotlib font cache を削除
cache_dir = matplotlib.get_cachedir()
shutil.rmtree(cache_dir, ignore_errors=True)

# 4. subprocess で font cache を再構築
# (約 10-15 秒かかります)
subprocess.run(["python", "-c", "import matplotlib.pyplot"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ setup 完了: font cache を再構築しました。!python script で Arial を使用できます。")

In [ ]:
# file upload 用 helper function
def upload_as(new_name: str):
    import os
    from google.colab import files
    print(f"次の file を upload してください: {new_name}")
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        os.rename(uploaded_name, new_name)
        print(f"-> {new_name} として保存しました\n")

In [ ]:
# input structures を upload
upload_as("react.xyz")
upload_as("prod.xyz")

### ▶ Run

In [ ]:
# full workflow を実行
!python molscout.py -r react.xyz -p prod.xyz -d result -c 0 -m orbmol

In [ ]:
# results を download
!zip -r /content/download.zip /content/result
from google.colab import files
files.download("/content/download.zip")

### ⚙ Experimental

### 🍣 sample_input の実行

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/HCN_HNC/reactant.xyz \
-p /content/MolScout/core/sample_input/HCN_HNC/product.xyz \
-d result/sample_01 -c 0 -m orbmol+alpb

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/Cl-_CH3F/reactant.xyz \
-p /content/MolScout/core/sample_input/Cl-_CH3F/product.xyz \
-d result/sample_02 -c -1 -m orbmol

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/NH3_ud/reactant.xyz \
-p /content/MolScout/core/sample_input/NH3_ud/product.xyz \
-d result/sample_03 -c 0 -m orbmol

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/formic_acid_dimer/reactant.xyz \
-p /content/MolScout/core/sample_input/formic_acid_dimer/product.xyz \
-d result/sample_04 -c 0 -m orbmol

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/bi_anthracene/reactant.xyz \
-p /content/MolScout/core/sample_input/bi_anthracene/product.xyz \
-d result/sample_05 -c 0 -m orbmol